# Hidden-unit sweep: standard RNN vs GRU — v8

Redoes the sweep with a **standard (Elman) RNN** to see whether it trains as well as the GRU. If it does,
the RNN is the better choice for the interpretability work, because its dynamics are far easier to analyse
(no gates).

Both architectures are swept over 1–8 hidden units × 5 seeds on the same v8 data and overlaid, so the
comparison is direct. Everything else (data, training, readout) is identical to `03_hidden_unit_sweep_v8`.

A vanilla RNN has to hold the answer in its state from the stimulus window (steps 10–20) to the readout at
step 50, a ~30-step delay, and plain RNNs are weaker than GRUs at that, so it may need a few more units or
train less cleanly at the smallest sizes. That gap is what is being measured.

## 1. Setup

In [ ]:
import os
os.environ["MKL_DISABLE_FAST_MM"] = "1"   # optional
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import time

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda"); print("Using GPU (CUDA):", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("Using GPU (MPS - Apple Silicon)")
else:
    device = torch.device("cpu"); print("WARNING: no GPU acceleration, using CPU (slower).")

DATA_DIR = Path("./generated_trials_v8")

MODEL_DIR = Path("./trained_models_v8_rnn_sweep"); MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
import torch
device = torch.device("cpu")
torch.set_num_threads(torch.get_num_threads())   # use all cores
print("Forcing device:", device)

## 2. Load data and subtask groups

In [ ]:
DET_SUBTASKS = ["det_absent","det_auditory_only","det_visual_only","det_multisensory"]
LOC_SUBTASKS = ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
                "loc_multisensory_same_L","loc_multisensory_same_R",
                "loc_conflict_audL_visR","loc_conflict_audR_visL"]
SUBTASKS = DET_SUBTASKS + LOC_SUBTASKS
TEST_ONLY = {"det_multisensory"}            # not trained
CONFLICT  = {"det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"}

# Colour each subtask by group: detection = oranges, localisation = blues.
_det_cols = plt.cm.Oranges(np.linspace(0.45, 0.92, len(DET_SUBTASKS)))
_loc_cols = plt.cm.Blues(np.linspace(0.35, 0.95, len(LOC_SUBTASKS)))
SUBTASK_STYLE = {}
for i, s in enumerate(DET_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_det_cols[i], ls=("--" if s in CONFLICT else "-"))
for i, s in enumerate(LOC_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_loc_cols[i], ls=("--" if s in CONFLICT else "-"))

def load_dataset(filename):
    d = np.load(DATA_DIR / filename, allow_pickle=True)
    return {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64),
            "types": d["types"], "n_classes": int(d["n_classes"])}

train = load_dataset("train.npz")
test  = load_dataset("test.npz")
X_tr, y_tr = train["X"], train["y"]
X_te, y_te, types_te = test["X"], test["y"], test["types"]
TEST_MASKS = {s: (types_te == s) for s in SUBTASKS}
print("Train:", X_tr.shape, " Test:", X_te.shape)
print("Train class balance:", np.bincount(y_tr, minlength=4))

## 3. Model with a GRU / RNN switch

In [ ]:
class UnifiedRNN(nn.Module):
    """Single recurrent layer + linear readout. cell = "gru" or "rnn" (Elman, tanh)."""
    def __init__(self, n_channels=4, hidden_size=8, n_classes=4, cell="rnn"):
        super().__init__()
        if cell == "gru":
            self.rnn = nn.GRU(n_channels, hidden_size, batch_first=True)
        elif cell == "rnn":
            self.rnn = nn.RNN(n_channels, hidden_size, batch_first=True, nonlinearity="tanh")
        else:
            raise ValueError(cell)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2)          # (B, channels, time) -> (B, time, channels)
        h, _ = self.rnn(x)
        return self.readout(h)         # (B, time, n_classes)

## 4. Train-one helper (same as the GRU sweep, with a cell argument)

In [ ]:
def train_one(hidden_size, seed, cell, n_epochs, lr, batch_size, device):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                        batch_size=batch_size, shuffle=True)
    X_test_t = torch.from_numpy(X_te).to(device)
    model = UnifiedRNN(4, hidden_size, 4, cell=cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb); B, T, C = logits.shape
            loss = loss_fn(logits.reshape(B*T, C), yb.unsqueeze(1).expand(B, T).reshape(B*T))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X_test_t)[:, -1, :].argmax(-1).cpu().numpy()
    overall = float((pred == y_te).mean())
    per_sub = {s: float((pred[TEST_MASKS[s]] == y_te[TEST_MASKS[s]]).mean()) for s in SUBTASKS}
    return overall, per_sub

## 5. Sweep config

Both architectures, 1–8 units, 5 seeds, so twice the work of the GRU-only sweep. This is the slow cell.

In [ ]:
HIDDEN_SIZES = list(range(1, 9))   # 1..8
SEEDS = [0, 1, 2, 3, 4]
ARCHS = ["gru", "rnn"]
N_EPOCHS = 50
BATCH_SIZE = 64
LR = 1e-3
print("Networks to train:", len(ARCHS)*len(HIDDEN_SIZES)*len(SEEDS),
      "(", len(ARCHS), "archs x", len(HIDDEN_SIZES), "sizes x", len(SEEDS), "seeds )")

## 6. Run the sweep for both architectures

In [ ]:
import time
sweeps = {a: {} for a in ARCHS}       # arch -> h -> {"overall":[...], "per_sub":{s:[...]}}
t0 = time.time()
for a in ARCHS:
    print("ARCH:", a.upper())
    for h in HIDDEN_SIZES:
        overalls = []; per = {s: [] for s in SUBTASKS}; ts = time.time()
        for seed in SEEDS:
            ov, ps = train_one(h, seed, a, N_EPOCHS, LR, BATCH_SIZE, device)
            overalls.append(ov)
            for s in SUBTASKS: per[s].append(ps[s])
        sweeps[a][h] = {"overall": overalls, "per_sub": per}
        print("  h=%d  overall mean=%.3f std=%.3f  (%.1fs)" %
              (h, np.mean(overalls), np.std(overalls), time.time()-ts))
print("Total sweep time: %.1f min" % ((time.time()-t0)/60))

## 7. Direct comparison: overall accuracy vs hidden units

In [ ]:
arch_style = {"gru": ("black", "o", "GRU"), "rnn": ("tab:green", "s", "standard RNN")}
fig, ax = plt.subplots(figsize=(9, 6))
for a in ARCHS:
    means = np.array([np.mean(sweeps[a][h]["overall"]) for h in HIDDEN_SIZES])
    stds  = np.array([np.std(sweeps[a][h]["overall"])  for h in HIDDEN_SIZES])
    col, mk, lab = arch_style[a]
    ax.errorbar(HIDDEN_SIZES, means, yerr=stds, marker=mk, lw=2.2, capsize=4, color=col, label=lab)
    ax.fill_between(HIDDEN_SIZES, means - stds, means + stds, color=col, alpha=0.12)
ax.axhline(0.25, color="gray", ls=":", alpha=0.6, label="4-class chance")
ax.set_xlabel("Number of hidden units"); ax.set_ylabel("Final test accuracy")
ax.set_xticks(HIDDEN_SIZES); ax.set_ylim(0, 1.05)
ax.set_title("Standard RNN vs GRU: overall test accuracy (5 seeds)")
ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"\n{'Hidden':>7} {'GRU mean':>10} {'GRU std':>9} {'RNN mean':>10} {'RNN std':>9} {'gap':>7}")
print("-" * 56)
for h in HIDDEN_SIZES:
    g, r = sweeps["gru"][h]["overall"], sweeps["rnn"][h]["overall"]
    print(f"{h:>7} {np.mean(g):>10.3f} {np.std(g):>9.3f} {np.mean(r):>10.3f} {np.std(r):>9.3f} {np.mean(g)-np.mean(r):>7.3f}")

## 8. Per-subtask accuracy for the standard RNN

Shows where (if anywhere) the RNN struggles: usually the conflicts and the held-out det_multisensory.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for s in SUBTASKS:
    m  = np.array([np.mean(sweeps["rnn"][h]["per_sub"][s]) for h in HIDDEN_SIZES])
    sd = np.array([np.std(sweeps["rnn"][h]["per_sub"][s])  for h in HIDDEN_SIZES])
    st = SUBTASK_STYLE[s]
    ax.plot(HIDDEN_SIZES, m, color=st["color"], ls=st["ls"], lw=1.8, label=s)
    ax.fill_between(HIDDEN_SIZES, m - sd, m + sd, color=st["color"], alpha=0.12)
ax.axhline(0.25, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Number of hidden units"); ax.set_ylabel("Final test accuracy")
ax.set_xticks(HIDDEN_SIZES); ax.set_ylim(0, 1.05)
ax.set_title("Standard RNN: per-subtask accuracy vs hidden units (mean +/- std, 5 seeds)")
ax.grid(alpha=0.3); ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 9. Verdict

In [ ]:
# Compare the two at the plateau (h >= 3) to decide whether the RNN is "similarly good".
plateau = [h for h in HIDDEN_SIZES if h >= 3]
g_plat = np.mean([np.mean(sweeps["gru"][h]["overall"]) for h in plateau])
r_plat = np.mean([np.mean(sweeps["rnn"][h]["overall"]) for h in plateau])
print("Plateau (h>=3) mean overall:  GRU %.3f   RNN %.3f   gap %.3f" % (g_plat, r_plat, g_plat - r_plat))
# smallest RNN size that gets within 0.03 of the GRU plateau
ok = [h for h in HIDDEN_SIZES if np.mean(sweeps["rnn"][h]["overall"]) >= g_plat - 0.03]
print("Smallest RNN size reaching GRU-plateau (within 0.03):", ok[0] if ok else "none in 1-8")
if g_plat - r_plat < 0.03:
    print("\nVerdict: the standard RNN trains essentially as well as the GRU -> switch to RNN for interpretability.")
elif g_plat - r_plat < 0.08:
    print("\nVerdict: RNN is slightly behind but close -> likely fine, maybe use one or two more units.")
else:
    print("\nVerdict: RNN lags noticeably -> not a straight swap for the GRU.")

import numpy as _np
_np.savez(MODEL_DIR / "rnn_vs_gru_sweep_v8.npz", sweeps=_np.array(sweeps, dtype=object),
          hidden_sizes=_np.array(HIDDEN_SIZES), seeds=_np.array(SEEDS))
print("Saved sweep results to", MODEL_DIR / "rnn_vs_gru_sweep_v8.npz")